# Lecture 9 — RNNs: classify surnames by language

In the sandbox you **stepped a single RNN cell by hand** and ran BPTT on toy
numbers. Now you'll train the real thing: an `nn.LSTM` that reads a surname one
character at a time and predicts which **language** it comes from — on the
classic *"names by language"* dataset, with a GPU.

The key idea this notebook makes concrete (it's the whole point of Lecture 9):
an RNN returns **two** things — the per-step outputs `out` *and* the final
hidden state `h_n`. For a **sequence → one label** task you classify from
**`h_n`** (the network's summary of the *whole* name), not from `out`.

> **Runtime → Change runtime type → T4 GPU** before you start. (It's quick even
> on CPU — names are short — but let's do it the real way.)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU. "
          "(This notebook is tiny, so CPU is fine too.)")


def accuracy(logits, targets):
    """Fraction correct given (N, C) logits and (N,) integer targets."""
    return (logits.argmax(dim=1) == targets).float().mean().item()


def plot_curves(history):
    """Plot train/val loss and val accuracy from a dict of lists."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(history["train_loss"], label="train loss")
    ax1.plot(history["val_loss"], label="val loss")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.legend(); ax1.set_title("Loss")
    ax2.plot(history["val_acc"], label="val acc", color="tab:green")
    ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.legend(); ax2.set_title("Val accuracy")
    plt.tight_layout(); plt.show()

## The data — surnames labelled by language

We download the classic dataset PyTorch ships for its character-RNN tutorial: a
zip of `data/names/<Language>.txt` files, one **surname per line**. We work at
the **character level** — a name is a sequence of letters, the label is its
language.

What's *real and messy* about it:

- **Heavily class-imbalanced.** Some languages (Russian, English) have thousands
  of names; others (Korean, Vietnamese) have a few dozen. A model can score high
  accuracy just by leaning on the big classes — we'll see that bite us.
- **Unicode & accents.** Names like *Ślusàrski* contain accented letters; we
  strip them to plain ASCII so every name uses the same small alphabet.
- **Short sequences.** Most names are 5–10 characters — perfect for a first RNN.

In [ ]:
import os, zipfile, urllib.request, glob, unicodedata, string

URL = "https://download.pytorch.org/tutorial/data.zip"
ZIP_PATH = "data.zip"
if not os.path.exists("data/names"):
    print("downloading", URL)
    urllib.request.urlretrieve(URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(".")
print("name files:", len(glob.glob("data/names/*.txt")))

# Alphabet: ASCII letters + a few punctuation chars that appear in names.
ALL_LETTERS = string.ascii_letters + " .,;'-"
N_LETTERS = len(ALL_LETTERS)


def to_ascii(s):
    """Strip accents/unicode down to our ASCII alphabet (Ślusàrski -> Slusarski)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn" and c in ALL_LETTERS
    )


# Read every <Language>.txt into {language: [names]}.
category_lines, all_categories = {}, []
for path in sorted(glob.glob("data/names/*.txt")):
    lang = os.path.splitext(os.path.basename(path))[0]
    with open(path, encoding="utf-8") as f:
        names = [to_ascii(line.strip()) for line in f if line.strip()]
    names = [n for n in names if n]  # drop any that emptied out after ASCII-ing
    category_lines[lang] = names
    all_categories.append(lang)

N_CATEGORIES = len(all_categories)
print(f"{N_CATEGORIES} languages, {N_LETTERS} characters in the alphabet")
print("languages:", all_categories)

In [ ]:
# Encode a name as a tensor of character indices, e.g. "Lee" -> [37, 4, 4].
char_to_idx = {ch: i for i, ch in enumerate(ALL_LETTERS)}


def name_to_tensor(name):
    """(L,) LongTensor of char indices for one name."""
    return torch.tensor([char_to_idx[c] for c in name], dtype=torch.long)


# Flatten to a list of (name, label) and make a reproducible train/val/test split.
data = [(name, all_categories.index(lang))
        for lang, names in category_lines.items() for name in names]

g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(data), generator=g).tolist()
data = [data[i] for i in perm]

n = len(data)
n_test = int(0.15 * n)
n_val = int(0.15 * n)
test_data = data[:n_test]
val_data = data[n_test:n_test + n_val]
train_data = data[n_test + n_val:]
print(f"total {n} names -> train {len(train_data)}, val {len(val_data)}, test {len(test_data)}")

## Look at the data

A few names per language, then the **class histogram** — this is where the
imbalance jumps out.

In [ ]:
import random
random.seed(0)

for lang in random.sample(all_categories, 6):
    sample = random.sample(category_lines[lang], min(5, len(category_lines[lang])))
    print(f"{lang:12s}: {', '.join(sample)}")

# Class histogram: how many names per language (sorted) — the imbalance.
counts = sorted(((len(v), k) for k, v in category_lines.items()), reverse=True)
sizes = [c for c, _ in counts]
labels = [k for _, k in counts]

plt.figure(figsize=(12, 4))
plt.bar(range(len(sizes)), sizes, color="tab:purple")
plt.xticks(range(len(sizes)), labels, rotation=60, ha="right")
plt.ylabel("# names"); plt.title("Names per language (class imbalance)")
plt.tight_layout(); plt.show()

print(f"\nbiggest: {labels[0]} ({sizes[0]})   smallest: {labels[-1]} ({sizes[-1]})"
      f"   ratio ~ {sizes[0] / sizes[-1]:.0f}x")

## Build the model — sequence → ONE label

This is the cell that mirrors what you built from scratch. The plan:

1. **Embed** each character index into a vector (`nn.Embedding`).
2. Run the sequence through an **`nn.LSTM`**.
3. The LSTM returns `out, (h_n, c_n)`:
   - `out` has shape `(L, H)` — the hidden state at **every** timestep,
   - `h_n` has shape `(num_layers, H)` — the **final** hidden state only.
4. For **many-to-one** classification we take **`h_n[-1]`** (the summary of the
   *whole* name) and feed it to a `Linear` → one score per language.

Read it — and try writing the `forward` yourself — before running.

In [ ]:
# 🔧 Your turn — this mirrors what you built from scratch; read it (and try writing it yourself) before running.
class NameClassifier(nn.Module):
    def __init__(self, n_letters, n_categories, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embed = nn.Embedding(n_letters, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_categories)

    def forward(self, x):
        # x: (L,) char indices for ONE name.
        emb = self.embed(x).unsqueeze(0)          # (1, L, embed_dim)
        out, (h_n, c_n) = self.lstm(emb)          # out: (1, L, H);  h_n: (1, 1, H)
        # MANY-TO-ONE: classify from the FINAL hidden state h_n, not the per-step out.
        last = h_n[-1]                            # (1, H) — summary of the whole name
        return self.fc(last)                      # (1, n_categories) logits


model = NameClassifier(N_LETTERS, N_CATEGORIES).to(device)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

## Train

The five steps you know — forward, loss, zero-grad, backward, step — wrapped in
an epoch loop. We feed names **one at a time** (each is a different length, so
no padding needed), and call `.to(device)` on the model and every input. Names
are short, so a few passes over the data finishes in well under a minute on a
T4.

In [ ]:
# 🔧 Your turn — the training loop (the 5 steps). Read it before running.
import time

EPOCHS = 4
opt = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(split):
    """Mean loss + accuracy over a (name, label) list."""
    model.eval()
    total_loss, all_logits, all_targets = 0.0, [], []
    for name, label in split:
        x = name_to_tensor(name).to(device)
        y = torch.tensor([label], device=device)
        logits = model(x)
        total_loss += loss_fn(logits, y).item()
        all_logits.append(logits)
        all_targets.append(y)
    logits = torch.cat(all_logits); targets = torch.cat(all_targets)
    return total_loss / len(split), accuracy(logits, targets)


history = {"train_loss": [], "val_loss": [], "val_acc": []}
g = torch.Generator().manual_seed(0)
for epoch in range(EPOCHS):
    model.train()
    t0 = time.time()
    running = 0.0
    order = torch.randperm(len(train_data), generator=g).tolist()
    for i in order:
        name, label = train_data[i]
        x = name_to_tensor(name).to(device)             # (L,) -> device
        y = torch.tensor([label], device=device)        # (1,) -> device
        logits = model(x)                                # 1. forward
        loss = loss_fn(logits, y)                        # 2. loss
        opt.zero_grad()                                  # 3. zero grads
        loss.backward()                                  # 4. backward (BPTT!)
        opt.step()                                       # 5. update
        running += loss.item()

    train_loss = running / len(train_data)
    val_loss, val_acc = evaluate(val_data)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(f"epoch {epoch + 1}/{EPOCHS}  train_loss {train_loss:.3f}  "
          f"val_loss {val_loss:.3f}  val_acc {val_acc:.3f}  ({time.time() - t0:.1f}s)")

### 🔧 Diagnose & fix the broken run *(optional side-quest)*

The cell below has a **planted bug** — it's isolated so it won't affect anything
above. A teammate wrote this `forward` and complains *"my model barely beats
just guessing the biggest class — it never really learns the name."*

**Your task:** read it, predict what goes wrong, then fix it.

*Hint:* what shape is `out`, and which timestep should a many-to-one classifier
read from? Is `out[0]` the summary of the whole name, or just the first letter?

In [ ]:
# 🔧 BROKEN — find and fix the bug (this cell is a sandbox; it doesn't touch the trained model above).
class BrokenClassifier(nn.Module):
    def __init__(self, n_letters, n_categories, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embed = nn.Embedding(n_letters, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_categories)

    def forward(self, x):
        emb = self.embed(x).unsqueeze(0)          # (1, L, embed_dim)
        out, (h_n, c_n) = self.lstm(emb)          # out: (1, L, H)
        # BUG: out[:, 0] is the FIRST timestep — it has only seen the first letter,
        # not the whole name. A many-to-one head must read the LAST step.
        summary = out[:, 0]                       # <-- fix me
        return self.fc(summary)


# Sanity check: does reading the first step really hurt? Train a couple epochs.
broken = BrokenClassifier(N_LETTERS, N_CATEGORIES).to(device)
opt_b = torch.optim.Adam(broken.parameters(), lr=0.005)
g = torch.Generator().manual_seed(0)
for _ in range(2):
    broken.train()
    for i in torch.randperm(len(train_data), generator=g).tolist():
        name, label = train_data[i]
        x = name_to_tensor(name).to(device)
        y = torch.tensor([label], device=device)
        loss = loss_fn(broken(x), y)
        opt_b.zero_grad(); loss.backward(); opt_b.step()

broken.eval()
with torch.no_grad():
    logits = torch.cat([broken(name_to_tensor(nm).to(device)) for nm, _ in val_data])
    targets = torch.tensor([lb for _, lb in val_data], device=device)

# How well can you do by always guessing the biggest class? (the bar to clear)
majority_acc = max(targets.bincount().tolist()) / len(targets)
# Seeing only the first letter, the broken model barely clears that bar — far below
# the real model's val acc above. Fix `out[:, 0]` -> `out[:, -1]` so it reads the
# WHOLE name, and re-run: accuracy jumps.
print("broken val acc:", round(accuracy(logits, targets), 3),
      f"  (only the 1st letter; majority-guess alone ~ {majority_acc:.3f})")

## Evaluate

Training curves, then test accuracy, then a **"predict the language of THIS
name"** demo. We also peek at **per-class** accuracy, because a single accuracy
number hides what the imbalance is doing.

In [ ]:
plot_curves(history)

test_loss, test_acc = evaluate(test_data)
print(f"test accuracy: {test_acc:.3f}   (random guessing ~ {1 / N_CATEGORIES:.3f})")

In [ ]:
@torch.no_grad()
def predict(name, k=3):
    """Top-k predicted languages for a raw name string."""
    model.eval()
    x = name_to_tensor(to_ascii(name)).to(device)
    probs = torch.softmax(model(x), dim=1)[0]
    top_p, top_i = probs.topk(k)
    print(f"{name}:")
    for p, i in zip(top_p.tolist(), top_i.tolist()):
        print(f"    {all_categories[i]:12s} {p:.2f}")


for nm in ["Nakamoto", "Schmidt", "Romanov", "ONeill", "Nguyen"]:
    predict(nm)

In [ ]:
# Per-class accuracy — fairer than one global number when classes are imbalanced.
import collections
correct = collections.Counter(); total = collections.Counter()
model.eval()
with torch.no_grad():
    for name, label in test_data:
        pred = model(name_to_tensor(name).to(device)).argmax(dim=1).item()
        total[label] += 1
        if pred == label:
            correct[label] += 1

rows = []
for lab in range(N_CATEGORIES):
    if total[lab]:
        rows.append((correct[lab] / total[lab], total[lab], all_categories[lab]))
rows.sort()
print("per-class accuracy (worst first):")
for acc, n_lab, lang in rows:
    print(f"    {lang:12s} acc {acc:.2f}   (n={n_lab})")
print("\nNotice: small classes often score far below the headline accuracy.")

## Experiment — your turn

**1. `out[-1]` vs `h_n` are the *same* vector.** The lecture claims that for a
single-layer LSTM the last per-step output equals the final hidden state. Prove
it on a real name with the cell below.

**2. Classify YOUR surname.** Add your own name (and a few friends') to the
`predict(...)` calls. Does it guess a plausible language?

**3. Beat the baseline + fill the ablation table.** Test accuracy above is your
baseline. Try changes and record what happens:

| change | test acc | notes |
|---|---|---|
| baseline (LSTM, hidden=64, 4 epochs) | _fill in_ | |
| `hidden_dim=128` | _fill in_ | bigger summary |
| `nn.RNN` instead of `nn.LSTM` | _fill in_ | plain RNN — better/worse? |
| `EPOCHS=8` | _fill in_ | does val keep improving? |

In [ ]:
# EXPERIMENT 1 — show out[-1] and h_n are literally the same vector.
model.eval()
x = name_to_tensor(to_ascii("Romanov")).to(device)
with torch.no_grad():
    emb = model.embed(x).unsqueeze(0)
    out, (h_n, c_n) = model.lstm(emb)
print("out[:, -1] shape:", tuple(out[:, -1].shape), " h_n[-1] shape:", tuple(h_n[-1].shape))
print("max abs difference:", (out[:, -1] - h_n[-1]).abs().max().item())
print("-> same tensor: classifying from out[-1] == classifying from h_n. "
      "(out also gives you the EARLIER steps, which h_n throws away.)")

## Reflect — report YOUR results

Answer these from the numbers *you* produced (not generic ones):

1. **What test accuracy did you get**, and how does it compare to random
   guessing (`1 / N_CATEGORIES`)?
2. **Which languages get confused / score worst** in the per-class table — and
   is the **class imbalance** to blame (do the small classes score lowest)?
3. **What is `h_n`** — in your own words, what information does that final
   hidden vector carry, and why is it the right thing to classify a *whole* name
   from (vs. `out[0]`)?
4. **Train–val gap:** did `val_loss` track `train_loss`, or did one keep
   dropping while the other flattened? What would you try if they diverged?